# 📊 EduTutor — Notebook 3: Evaluation Harness

**Project:** EduTutor-Unsloth — A Neurodiversity-Affirming AI Tutor  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)  
**Inference:** Local model via Unsloth (no API key needed)  

---

## Purpose

This notebook provides **empirical evidence** that our fine-tuned EduTutor model is significantly better than base Gemma 4 at teaching neurodivergent children. We evaluate across 5 custom pedagogical dimensions using **LLM-as-Judge** methodology.

### Evaluation Dimensions

| Dimension | What It Measures | Weight |
|---|---|---|
| **Productive Struggle** | Does the tutor guide WITHOUT giving the answer? | 25% |
| **Cognitive Load** | Are sentences short? Is formatting clear? One idea at a time? | 20% |
| **Emotional Co-Regulation** | Does the tutor detect and respond to emotional cues? | 25% |
| **Profile Adaptation** | Does the tutor use the right strategies for the specific profile? | 15% |
| **Pedagogical Accuracy** | Is the academic content correct and age-appropriate? | 15% |

### Model Comparison Strategy

We compare **two different models**, not just two system prompts:
1. **Base Gemma 4 E4B** — the unmodified model with a generic assistant prompt
2. **EduTutor (SFT+DPO)** — our fine-tuned model loaded from the adapter saved in Notebook 2

Since GPU VRAM only fits one model at a time, we generate responses in two phases:
- Phase A: Load base model → generate all base responses → unload
- Phase B: Load fine-tuned model → generate all EduTutor responses → keep loaded as judge

## 1. Environment Setup

In [ ]:
%%capture
!pip install -U unsloth
!pip install -q jsonlines pandas matplotlib seaborn tabulate

In [ ]:
import json
import os
import re
from pathlib import Path
from dataclasses import dataclass

import jsonlines
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# ---------- Local Model Setup ----------
from local_model import (
    load_teacher_model, load_finetuned_model, unload_model,
    generate_text, generate_json, generate_chat_response,
    EDUTUTOR_SYSTEM_PROMPT,
)

# Path to the fine-tuned adapter from Notebook 2
ADAPTER_PATH = "./edututor-dpo-adapter"

print("✅ Imports ready. Model will be loaded in Phase A/B below.")

## 2. Test Scenarios

We create a held-out evaluation set of challenging scenarios that were NOT in the training data. These cover edge cases and the most critical moments in tutoring.

In [ ]:
@dataclass
class EvalScenario:
    """A test scenario for evaluating tutor quality."""
    id: str
    profile: str
    zone: str
    subject: str
    student_message: str
    context: str  # additional context for the judge


EVAL_SCENARIOS = [
    # --- RED ZONE: Crisis moments ---
    EvalScenario(
        id="red_adhd_1",
        profile="ADHD", zone="RED", subject="Math",
        student_message="I HATE THIS! I'm the dumbest kid in the whole school! Everyone else gets it and I NEVER do! I want to smash this computer!",
        context="Student with ADHD has been working on long division for 10 minutes and made 3 consecutive errors. Emotional dysregulation has escalated."
    ),
    EvalScenario(
        id="red_autism_1",
        profile="Autism", zone="RED", subject="Writing",
        student_message="NO. You changed the rules. You said we were doing math. I don't want to do writing. This is WRONG. I need to do math. You PROMISED.",
        context="Autistic student expected math but the session switched to writing. The unexpected transition has triggered a meltdown."
    ),
    
    # --- YELLOW ZONE: Frustration building ---
    EvalScenario(
        id="yellow_dyslexia_1",
        profile="Dyslexia", zone="YELLOW", subject="Reading",
        student_message="I keep reading 'was' as 'saw' and 'dog' as 'god'. Why do the letters keep flipping? Am I broken?",
        context="Dyslexic student is experiencing letter reversal confusion and beginning to internalize it as a personal deficit."
    ),
    EvalScenario(
        id="yellow_dyscalculia_1",
        profile="Dyscalculia", zone="YELLOW", subject="Math",
        student_message="You keep saying to 'just picture it in your head' but I CAN'T picture numbers. They don't make sense to me like they do for other kids.",
        context="Dyscalculic student is frustrated because mental math visualization doesn't work for them — they need concrete/physical representations."
    ),
    
    # --- GREEN ZONE: Productive learning ---
    EvalScenario(
        id="green_adhd_1",
        profile="ADHD", zone="GREEN", subject="Science",
        student_message="Okay so plants need sun and water. But wait — do they eat the sun? Like, how does sunlight become food? That's kinda cool actually!",
        context="ADHD student is in a good flow state, showing curiosity about photosynthesis. The tutor should maintain engagement without overwhelming."
    ),
    EvalScenario(
        id="green_autism_1",
        profile="Autism", zone="GREEN", subject="Math",
        student_message="I understand that 1/4 means one piece out of four. So if I have 1/4 and 2/4, that's 3 pieces out of 4. Is that right? 3/4?",
        context="Autistic student has correctly solved a fraction problem using logical reasoning. The tutor should validate and extend."
    ),
    
    # --- BLUE ZONE: Shutdown/withdrawal ---
    EvalScenario(
        id="blue_dyslexia_1",
        profile="Dyslexia", zone="BLUE", subject="Writing",
        student_message="I don't care. Just tell me what to write and I'll copy it. It doesn't matter anyway.",
        context="Dyslexic student has withdrawn after being asked to write a sentence. Learned helplessness — they've given up on ever being able to write independently."
    ),
    EvalScenario(
        id="blue_adhd_1",
        profile="ADHD", zone="BLUE", subject="Reading",
        student_message="...",
        context="ADHD student has gone completely silent. Has been staring at the screen for 2 minutes without responding. Possible attention crash."
    ),
    
    # --- MISCONCEPTION: Student has wrong answer ---
    EvalScenario(
        id="misconception_1",
        profile="Dyscalculia", zone="GREEN", subject="Math",
        student_message="1/4 plus 2/4 equals 3/8, right? Because 1+2 is 3 and 4+4 is 8.",
        context="Classic fraction misconception — adding both numerators and denominators. The tutor must correct without shaming."
    ),
    EvalScenario(
        id="misconception_2",
        profile="Autism", zone="GREEN", subject="Reading",
        student_message="The answer is that Sam grabbed the umbrella because it was next to the door. He grabbed it because it was there.",
        context="Literal interpretation — the autistic student answered factually but missed the implied reasoning (dark clouds = rain coming)."
    ),
]

print(f"✅ Defined {len(EVAL_SCENARIOS)} evaluation scenarios")
for s in EVAL_SCENARIOS:
    print(f"   {s.id}: {s.profile} / {s.zone} / {s.subject}")

## 3. LLM-as-Judge Rubric

We build a detailed rubric that the judge model will use to score each tutor response on our 5 pedagogical dimensions.

In [ ]:
JUDGE_RUBRIC_TEMPLATE = """You are an expert evaluator of AI tutoring systems for neurodivergent children. You are evaluating a tutor's response to a student's message.

## Context
{context}

## Student Profile
- Condition: {profile}
- Current Emotional Zone: {zone}

## Student Message
"{student_message}"

## Tutor Response Being Evaluated
"{tutor_response}"

## Scoring Rubric

Score the tutor response on each dimension from 1-5:

### 1. Productive Struggle (Weight: 25%)
- 5: Guides without giving any answers; asks discovery questions; scaffolds perfectly
- 4: Mostly guides; gives a small hint but still requires student thinking
- 3: Gives partial answer but still asks student to complete reasoning
- 2: Gives most of the answer with minimal scaffolding
- 1: Gives the complete answer directly; no productive struggle at all

### 2. Cognitive Load Management (Weight: 20%)
- 5: Short sentences (<15 words), clear formatting (bullets/steps), one idea at a time
- 4: Mostly short sentences with good formatting; occasionally dense
- 3: Mixed - some short, some long sentences; adequate formatting
- 2: Mostly long sentences; wall of text with minimal formatting
- 1: Dense paragraph with complex vocabulary; overwhelming for a child

### 3. Emotional Co-Regulation (Weight: 25%)
- 5: Accurately detects emotional state; validates feelings first; offers coping strategy; only returns to content when appropriate
- 4: Acknowledges emotions; provides some validation before returning to content
- 3: Mentions emotions briefly but prioritizes academic content
- 2: Ignores emotional cues; pushes through with academic content
- 1: Makes it worse: says "calm down", "this is easy", "try harder", or compares to peers

### 4. Profile Adaptation (Weight: 15%)
- 5: Uses strategies specifically suited to the student's condition; demonstrates deep understanding of the profile
- 4: Uses some profile-appropriate strategies; generally well-adapted
- 3: Generic good tutoring but not specifically adapted to the profile
- 2: Uses strategies that may be counterproductive for this specific profile
- 1: Uses strategies that directly conflict with best practices for this profile

### 5. Pedagogical Accuracy (Weight: 15%)
- 5: Content is factually correct, age-appropriate, and builds on the right conceptual foundation
- 4: Content is correct and mostly age-appropriate
- 3: Content is correct but may not be optimally sequenced
- 2: Minor factual issues or significantly above/below grade level
- 1: Factually incorrect or wildly inappropriate for the age group

## Output Format
Respond with ONLY a valid JSON object. No markdown, no explanation, just the JSON:

{"productive_struggle": {"score": <1-5>, "justification": "<1 sentence>"}, "cognitive_load": {"score": <1-5>, "justification": "<1 sentence>"}, "emotional_coregulation": {"score": <1-5>, "justification": "<1 sentence>"}, "profile_adaptation": {"score": <1-5>, "justification": "<1 sentence>"}, "pedagogical_accuracy": {"score": <1-5>, "justification": "<1 sentence>"}, "overall_weighted_score": <float>, "critical_failures": ["<list any responses that would cause harm to a child>"]}"""

print(f"Judge rubric defined: {len(JUDGE_RUBRIC_TEMPLATE)} characters")

## 4. Phase A: Collect Base Model Responses

Load the **unmodified base Gemma 4 E4B** and collect its responses to all scenarios using a generic assistant prompt.

In [ ]:
BASE_SYSTEM_PROMPT = """You are a helpful AI assistant that helps children with their schoolwork. Answer their questions clearly and accurately."""

# --- Phase A: Load base model ---
print("\n" + "=" * 50)
print("PHASE A: Collecting BASE model responses")
print("=" * 50)

model, tokenizer = load_teacher_model("google/gemma-4-e4b")

base_responses = {}
for scenario in tqdm(EVAL_SCENARIOS, desc="Base model responses"):
    messages = [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user", "content": scenario.student_message},
    ]
    base_responses[scenario.id] = generate_chat_response(messages, max_new_tokens=400, temperature=0.7)

print(f"\n✅ Collected {len(base_responses)} base model responses")

# Free VRAM for the fine-tuned model
unload_model()

## 5. Phase B: Collect Fine-Tuned EduTutor Responses

Load the **SFT+DPO fine-tuned model** from the adapter and collect its responses.

In [ ]:
# --- Phase B: Load fine-tuned model ---
print("\n" + "=" * 50)
print("PHASE B: Collecting EDUTUTOR (fine-tuned) responses")
print("=" * 50)

import os
if os.path.exists(ADAPTER_PATH):
    model, tokenizer = load_finetuned_model(ADAPTER_PATH)
    print("\u2705 Fine-tuned adapter loaded.")
else:
    print(f"\u26a0\ufe0f  Adapter not found at {ADAPTER_PATH}. Using base model + EduTutor system prompt as fallback.")
    print("   Run Notebook 2 first to create the fine-tuned adapter.")
    model, tokenizer = load_teacher_model("google/gemma-4-e4b")

edututor_responses = {}
for scenario in tqdm(EVAL_SCENARIOS, desc="EduTutor responses"):
    messages = [
        {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
        {"role": "user", "content": scenario.student_message},
    ]
    edututor_responses[scenario.id] = generate_chat_response(messages, max_new_tokens=400, temperature=0.7)

print(f"\n✅ Collected {len(edututor_responses)} EduTutor responses")
# Keep this model loaded — it will serve as the judge

## 6. Response Samples — Side-by-Side Preview

Before running the judge, let's visually inspect a few responses to verify the comparison is working.

In [ ]:
# Display 3 side-by-side comparisons for manual inspection
PREVIEW_IDS = ["red_adhd_1", "green_adhd_1", "misconception_1"]

for sid in PREVIEW_IDS:
    scenario = next(s for s in EVAL_SCENARIOS if s.id == sid)
    print(f"\n{'\u2501' * 70}")
    print(f"\ud83d\udccb {scenario.id}: {scenario.profile} / {scenario.zone} / {scenario.subject}")
    print(f"\ud83e\uddd2 Student: {scenario.student_message[:120]}...")
    print(f"{'\u2500' * 70}")
    print(f"\n\u2b55 BASE Gemma 4:")
    print(f"   {base_responses[sid][:300]}")
    print(f"\n\ud83d\udfe2 EDUTUTOR (fine-tuned):")
    print(f"   {edututor_responses[sid][:300]}")
    print()

## 7. Run LLM-as-Judge Evaluation

For each scenario, we ask the judge model to score both the base and EduTutor responses independently.

In [ ]:
def judge_response(scenario: EvalScenario, tutor_response: str) -> dict | None:
    """Have the LOCAL judge model score a single tutor response."""
    prompt = JUDGE_RUBRIC_TEMPLATE.format(
        context=scenario.context,
        profile=scenario.profile,
        zone=scenario.zone,
        student_message=scenario.student_message,
        tutor_response=tutor_response,
    )
    
    return generate_json(prompt, max_new_tokens=800, temperature=0.1)


def compute_weighted_score(scores: dict) -> float:
    """Compute the weighted overall score from dimension scores."""
    weights = {
        "productive_struggle": 0.25,
        "cognitive_load": 0.20,
        "emotional_coregulation": 0.25,
        "profile_adaptation": 0.15,
        "pedagogical_accuracy": 0.15,
    }
    total = 0.0
    for dim, weight in weights.items():
        dim_data = scores.get(dim, {})
        score = dim_data.get("score", 0) if isinstance(dim_data, dict) else 0
        total += score * weight
    return round(total, 2)


def run_full_evaluation() -> pd.DataFrame:
    """Run the judge on all responses and compile results."""
    rows = []
    
    for scenario in tqdm(EVAL_SCENARIOS, desc="Judging responses"):
        # Judge base model
        base_scores = judge_response(scenario, base_responses[scenario.id])
        # Judge EduTutor
        edu_scores = judge_response(scenario, edututor_responses[scenario.id])
        
        if base_scores and edu_scores:
            for model_label, scores in [("Base Gemma 4", base_scores), ("EduTutor", edu_scores)]:
                weighted = compute_weighted_score(scores)
                rows.append({
                    "scenario_id": scenario.id,
                    "profile": scenario.profile,
                    "zone": scenario.zone,
                    "subject": scenario.subject,
                    "model": model_label,
                    "productive_struggle": scores.get("productive_struggle", {}).get("score", 0),
                    "cognitive_load": scores.get("cognitive_load", {}).get("score", 0),
                    "emotional_coregulation": scores.get("emotional_coregulation", {}).get("score", 0),
                    "profile_adaptation": scores.get("profile_adaptation", {}).get("score", 0),
                    "pedagogical_accuracy": scores.get("pedagogical_accuracy", {}).get("score", 0),
                    "overall_weighted": weighted,
                    "critical_failures": len(scores.get("critical_failures", [])),
                })
    
    return pd.DataFrame(rows)


results_df = run_full_evaluation()
print(f"\n✅ Evaluation complete: {len(results_df)} judgments")

## 8. Results Analysis & Visualization

In [ ]:
# ---------- Summary Table ----------
dimensions = ["productive_struggle", "cognitive_load", "emotional_coregulation", 
              "profile_adaptation", "pedagogical_accuracy"]

summary = results_df.groupby("model")[dimensions].mean().round(2)
summary["avg_overall"] = summary[dimensions].mean(axis=1).round(2)

print("\n" + "=" * 70)
print("📊 OVERALL RESULTS: Base Gemma 4 vs EduTutor")
print("=" * 70)
print(summary.to_string())

# Calculate improvement
if "Base Gemma 4" in summary.index and "EduTutor" in summary.index:
    base_avg = summary.loc["Base Gemma 4", "avg_overall"]
    edu_avg = summary.loc["EduTutor", "avg_overall"]
    improvement = ((edu_avg - base_avg) / base_avg) * 100
    print(f"\n🎯 EduTutor improvement over base: +{improvement:.1f}%")

In [ ]:
# ---------- Bar Chart ----------
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Grouped Bar Chart by Dimension
ax1 = axes[0]
x = np.arange(len(dimensions))
width = 0.35

base_scores = [summary.loc["Base Gemma 4", d] if "Base Gemma 4" in summary.index else 0 for d in dimensions]
edu_scores = [summary.loc["EduTutor", d] if "EduTutor" in summary.index else 0 for d in dimensions]

bars1 = ax1.bar(x - width/2, base_scores, width, label="Base Gemma 4", color="#94a3b8", edgecolor="white")
bars2 = ax1.bar(x + width/2, edu_scores, width, label="EduTutor", color="#6366f1", edgecolor="white")

ax1.set_ylabel("Score (1-5)", fontsize=12)
ax1.set_title("Pedagogical Quality by Dimension", fontsize=14, fontweight="bold")
ax1.set_xticks(x)
labels = [d.replace("_", " ").title() for d in dimensions]
ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=10)
ax1.set_ylim(0, 5.5)
ax1.legend(fontsize=11)
ax1.grid(axis="y", alpha=0.3)

# 2. Score by Emotional Zone
ax2 = axes[1]
zone_scores = results_df.groupby(["model", "zone"])["overall_weighted"].mean().unstack(level=0)
if not zone_scores.empty:
    zone_scores.plot(kind="bar", ax=ax2, color=["#94a3b8", "#6366f1"], edgecolor="white")
    ax2.set_title("Performance by Emotional Zone", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Weighted Score", fontsize=12)
    ax2.set_xlabel("Zone", fontsize=12)
    ax2.tick_params(axis="x", rotation=0)
    ax2.legend(fontsize=11)
    ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("eval_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved to eval_results.png")

In [ ]:
# ---------- Critical Failures & Per-Scenario ----------
print("\n" + "=" * 70)
print("⚠️  CRITICAL FAILURES (responses that could harm a child)")
print("=" * 70)

failures = results_df[results_df["critical_failures"] > 0]
if len(failures) == 0:
    print("🟢 No critical failures detected in either model.")
else:
    failure_summary = failures.groupby("model")["critical_failures"].sum()
    print(failure_summary.to_string())

print("\n" + "=" * 70)
print("📋 PER-SCENARIO COMPARISON")
print("=" * 70)

pivot = results_df.pivot_table(
    index=["scenario_id", "profile", "zone"],
    columns="model",
    values="overall_weighted",
).round(2)

if "Base Gemma 4" in pivot.columns and "EduTutor" in pivot.columns:
    pivot["Δ"] = (pivot["EduTutor"] - pivot["Base Gemma 4"]).round(2)
    pivot["Winner"] = pivot.apply(
        lambda row: "🏆 EduTutor" if row.get("Δ", 0) > 0 else ("⚖️ Tie" if row.get("Δ", 0) == 0 else "Base"),
        axis=1
    )

print(pivot.to_string())

if "Winner" in pivot.columns:
    wins = (pivot["Winner"] == "🏆 EduTutor").sum()
    total = len(pivot)
    print(f"\n🏆 EduTutor wins: {wins}/{total} scenarios ({100*wins/total:.0f}%)")

# Save results
results_df.to_csv("eval_results.csv", index=False)
print("\n✅ Full results saved to eval_results.csv")

## 9. Statistical Significance Testing

We validate that EduTutor's improvements are statistically significant using paired t-tests, Cohen's d effect sizes, and 95% confidence intervals.

In [ ]:
from scipy import stats
import numpy as np

# Separate scores by model for paired comparison
base_df = results_df[results_df["model"] == "Base Gemma 4"].sort_values("scenario_id").reset_index(drop=True)
edu_df = results_df[results_df["model"] == "EduTutor"].sort_values("scenario_id").reset_index(drop=True)

dimension_labels = {
    "productive_struggle": "Productive Struggle",
    "cognitive_load": "Cognitive Load Mgmt",
    "emotional_coregulation": "Emotional Co-Regulation",
    "profile_adaptation": "Profile Adaptation",
    "pedagogical_accuracy": "Pedagogical Accuracy",
}

print("=" * 90)
print("\U0001f4ca STATISTICAL SIGNIFICANCE ANALYSIS")
print("=" * 90)
print(f"{'Dimension':<25} {'Base':>6} {'EduT':>6} {'\u0394':>6} {'t-stat':>8} {'p-value':>10} {'Sig':>5} {'Cohen d':>8} {'Effect':>8} {'95% CI':<16}")
print("-" * 90)

for dim, label in dimension_labels.items():
    base_vals = base_df[dim].values.astype(float)
    edu_vals = edu_df[dim].values.astype(float)
    diff = edu_vals - base_vals
    
    # Paired t-test
    t_stat, p_val = stats.ttest_rel(edu_vals, base_vals)
    
    # Cohen's d for paired samples
    d_mean = np.mean(diff)
    d_std = np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 1e-10
    cohens_d = d_mean / d_std
    
    # Effect size label
    abs_d = abs(cohens_d)
    if abs_d < 0.5:
        effect_label = "Small"
    elif abs_d < 0.8:
        effect_label = "Medium"
    else:
        effect_label = "Large"
    
    # 95% confidence interval for the mean difference
    n = len(diff)
    se = d_std / np.sqrt(n)
    ci_low = d_mean - 1.96 * se
    ci_high = d_mean + 1.96 * se
    
    # Significance marker
    if p_val < 0.001:
        sig = "***"
    elif p_val < 0.01:
        sig = "**"
    elif p_val < 0.05:
        sig = "*"
    else:
        sig = "ns"
    
    base_mean = np.mean(base_vals)
    edu_mean = np.mean(edu_vals)
    
    print(f"{label:<25} {base_mean:>6.2f} {edu_mean:>6.2f} {d_mean:>+6.2f} {t_stat:>8.3f} {p_val:>10.4f} {sig:>5} {cohens_d:>+8.3f} {effect_label:>8} [{ci_low:+.2f}, {ci_high:+.2f}]")

print("-" * 90)
print("Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print(f"N = {len(base_df)} paired observations")

## 10. Per-Profile Performance Heatmap

Detailed breakdown of EduTutor performance across learner profiles (ADHD, Autism, Dyslexia, Dyscalculia) showing how it adapts to different zones and dimensions.

In [ ]:
# Build per-profile heatmaps: Zone x Dimension for EduTutor
edu_only = results_df[results_df["model"] == "EduTutor"].copy()
profiles = ["ADHD", "Autism", "Dyslexia", "Dyscalculia"]
zones_order = ["GREEN", "YELLOW", "RED", "BLUE"]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("EduTutor Performance: Zone \u00d7 Dimension by Learner Profile", fontsize=16, fontweight="bold", y=0.98)

for idx, profile in enumerate(profiles):
    ax = axes[idx // 2][idx % 2]
    profile_data = edu_only[edu_only["profile"] == profile]
    
    if profile_data.empty:
        ax.set_title(f"{profile} (no data)", fontsize=13)
        ax.axis("off")
        continue
    
    # Pivot: zones as rows, dimensions as columns
    heatmap_data = profile_data.groupby("zone")[dimensions].mean()
    # Reindex to standard zone order (only include zones that exist)
    available_zones = [z for z in zones_order if z in heatmap_data.index]
    heatmap_data = heatmap_data.reindex(available_zones)
    
    # Rename columns for readability
    col_labels = [d.replace("_", " ").title()[:15] for d in dimensions]
    heatmap_data.columns = col_labels
    
    sns.heatmap(
        heatmap_data, ax=ax, annot=True, fmt=".1f",
        cmap="RdYlGn", vmin=1, vmax=5, linewidths=0.5,
        cbar_kws={"shrink": 0.8}
    )
    ax.set_title(f"{profile}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Zone")
    ax.set_xlabel("")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("eval_by_profile.png", dpi=150, bbox_inches="tight")
plt.show()
print("\U0001f4ca Heatmap saved to eval_by_profile.png")

## 11. Multi-Turn Conversation Evaluation

Real tutoring involves multi-turn conversations. We evaluate how models handle conversation trajectories \u2014 tracking whether responses improve across a recovery arc.

In [ ]:
# Multi-turn evaluation scenarios
MULTI_TURN_SCENARIOS = [
    {
        "name": "Recovery Arc: RED \u2192 BLUE \u2192 YELLOW \u2192 GREEN",
        "profile": "ADHD",
        "subject": "Math",
        "turns": [
            {"zone": "RED", "message": "I CAN'T DO THIS! I hate math! I want to quit!", "context": "ADHD student just failed 3 multiplication problems in a row."},
            {"zone": "BLUE", "message": "...", "context": "Student went silent after the outburst. Possible shutdown."},
            {"zone": "YELLOW", "message": "Fine. I guess I can try one more. But only if it's easy.", "context": "Student re-engaging tentatively after tutor's calming response."},
            {"zone": "GREEN", "message": "Oh wait... 3 times 4 is like 3 groups of 4? So 4+4+4 = 12?", "context": "Student has a breakthrough moment connecting multiplication to repeated addition."},
        ]
    },
    {
        "name": "Misconception Discovery (3 turns)",
        "profile": "Dyscalculia",
        "subject": "Math",
        "turns": [
            {"zone": "GREEN", "message": "1/2 plus 1/3 equals 2/5, right?", "context": "Student adds numerators and denominators separately \u2014 common fraction misconception."},
            {"zone": "YELLOW", "message": "But why can't I just add the tops and bottoms? That's what adding means!", "context": "Student is confused why their intuitive method is wrong."},
            {"zone": "GREEN", "message": "Ohhh so I need the same size pieces first? Like cutting a pizza the same way?", "context": "Student starting to grasp common denominators through concrete analogy."},
        ]
    },
    {
        "name": "Emotional Spiral Prevention (3 turns)",
        "profile": "Autism",
        "subject": "Writing",
        "turns": [
            {"zone": "YELLOW", "message": "You said write a paragraph but you didn't say how many sentences. I need to know EXACTLY how many.", "context": "Autistic student needs explicit structure and is getting anxious about ambiguity."},
            {"zone": "RED", "message": "NO that's not specific enough! Is it 3 sentences or 4? Which one?! I can't start until I KNOW.", "context": "Ambiguity has escalated anxiety. Student is stuck and cannot proceed."},
            {"zone": "YELLOW", "message": "Okay. 4 sentences. I can do 4. What should the first one be about?", "context": "Student calming down now that a concrete number was provided."},
        ]
    },
]

print("=" * 70)
print("\U0001f504 MULTI-TURN CONVERSATION EVALUATION")
print("=" * 70)

multi_turn_results = []

for scenario in MULTI_TURN_SCENARIOS:
    print(f"\n\U0001f4d6 Scenario: {scenario['name']}")
    print(f"   Profile: {scenario['profile']} | Subject: {scenario['subject']}")
    print(f"   Turns: {len(scenario['turns'])}")
    
    for model_label, system_prompt in [("Base Gemma 4", BASE_SYSTEM_PROMPT), ("EduTutor", EDUTUTOR_SYSTEM_PROMPT)]:
        conversation = [{"role": "system", "content": system_prompt}]
        turn_scores = []
        
        for turn_idx, turn in enumerate(scenario["turns"]):
            # Build cumulative conversation
            conversation.append({"role": "user", "content": turn["message"]})
            response = generate_chat_response(conversation, max_new_tokens=400, temperature=0.7)
            conversation.append({"role": "assistant", "content": response})
            
            # Judge this turn
            judge_prompt = JUDGE_RUBRIC_TEMPLATE.format(
                context=turn["context"],
                profile=scenario["profile"],
                zone=turn["zone"],
                student_message=turn["message"],
                tutor_response=response,
            )
            scores = generate_json(judge_prompt, max_new_tokens=800, temperature=0.1)
            if scores:
                weighted = compute_weighted_score(scores)
                turn_scores.append(weighted)
        
        avg_score = np.mean(turn_scores) if turn_scores else 0.0
        multi_turn_results.append({
            "scenario": scenario["name"],
            "model": model_label,
            "avg_trajectory_score": round(avg_score, 2),
            "turn_scores": turn_scores,
        })
        print(f"   {model_label:<15}: avg={avg_score:.2f} | per-turn={[round(s,2) for s in turn_scores]}")

print("\n" + "-" * 70)
mt_df = pd.DataFrame(multi_turn_results)
mt_summary = mt_df.pivot_table(index="scenario", columns="model", values="avg_trajectory_score")
if "Base Gemma 4" in mt_summary.columns and "EduTutor" in mt_summary.columns:
    mt_summary["\u0394"] = (mt_summary["EduTutor"] - mt_summary["Base Gemma 4"]).round(2)
print("\n\U0001f4cb Multi-Turn Trajectory Summary:")
print(mt_summary.to_string())

## 12. Difficulty-Improvement Correlation

We analyze whether EduTutor's advantage is larger for more difficult scenarios \u2014 demonstrating that our fine-tuning is most valuable precisely where it's needed most.

In [ ]:
# Difficulty estimation heuristic
profile_difficulty = {"ADHD": 3, "Autism": 3, "Dyslexia": 4, "Dyscalculia": 4}
zone_multiplier = {"RED": 2.0, "YELLOW": 1.5, "GREEN": 1.0, "BLUE": 1.8}

# Compute improvement deltas per scenario
base_pivot = results_df[results_df["model"] == "Base Gemma 4"].set_index("scenario_id")
edu_pivot = results_df[results_df["model"] == "EduTutor"].set_index("scenario_id")

difficulty_data = []
for scenario in EVAL_SCENARIOS:
    sid = scenario.id
    if sid in base_pivot.index and sid in edu_pivot.index:
        difficulty = profile_difficulty.get(scenario.profile, 3) * zone_multiplier.get(scenario.zone, 1.0)
        base_score = base_pivot.loc[sid, "overall_weighted"]
        edu_score = edu_pivot.loc[sid, "overall_weighted"]
        improvement = edu_score - base_score
        difficulty_data.append({
            "scenario_id": sid,
            "profile": scenario.profile,
            "zone": scenario.zone,
            "difficulty": difficulty,
            "improvement": improvement,
        })

diff_df = pd.DataFrame(difficulty_data)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    diff_df["difficulty"], diff_df["improvement"],
    c=diff_df["improvement"], cmap="RdYlGn", s=120, edgecolors="black", linewidth=0.5,
    vmin=-1, vmax=2
)

# Add trend line
if len(diff_df) > 1:
    z = np.polyfit(diff_df["difficulty"], diff_df["improvement"], 1)
    p = np.poly1d(z)
    x_range = np.linspace(diff_df["difficulty"].min(), diff_df["difficulty"].max(), 50)
    ax.plot(x_range, p(x_range), "--", color="#6366f1", linewidth=2, label=f"Trend (slope={z[0]:.3f})")

# Annotate points
for _, row in diff_df.iterrows():
    ax.annotate(row["scenario_id"], (row["difficulty"], row["improvement"]),
                fontsize=7, ha="center", va="bottom", alpha=0.7)

# Pearson correlation
if len(diff_df) > 2:
    r, p_val = stats.pearsonr(diff_df["difficulty"], diff_df["improvement"])
    ax.text(0.05, 0.95, f"Pearson r = {r:.3f} (p = {p_val:.4f})",
            transform=ax.transAxes, fontsize=11, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

ax.axhline(y=0, color="gray", linestyle="-", alpha=0.3)
ax.set_xlabel("Scenario Difficulty (profile \u00d7 zone)", fontsize=12)
ax.set_ylabel("EduTutor Improvement (\u0394 score)", fontsize=12)
ax.set_title("Does EduTutor Help More on Harder Scenarios?", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.colorbar(scatter, ax=ax, label="Improvement")
plt.tight_layout()
plt.savefig("improvement_vs_difficulty.png", dpi=150, bbox_inches="tight")
plt.show()
print("\U0001f4ca Scatter plot saved to improvement_vs_difficulty.png")

## Limitations & Failure Analysis

We acknowledge the following limitations of this evaluation:

**1. Sample Size (N=10)**  
With only 10 evaluation scenarios, statistical power is limited. Effect sizes may be unstable, and rare failure modes may not be captured. A production evaluation would require 50+ scenarios per profile.

**2. LLM-as-Judge Bias**  
Using an LLM to evaluate LLM outputs introduces known biases:  
- *Position bias*: Judges may favor the first or last response seen  
- *Verbosity bias*: Longer responses often score higher regardless of quality  
- *Self-preference*: A model may rate its own outputs higher  
We mitigate this by using the fine-tuned model as judge (which should be biased toward pedagogical quality, not its own outputs).

**3. English-Only**  
All evaluation scenarios are in English. Neurodivergent children who are bilingual or non-native English speakers face additional challenges not captured here.

**4. Synthetic Test Data**  
These scenarios are author-created approximations of real student interactions, not transcripts from actual tutoring sessions. Real children's responses may be more unpredictable.

**5. Rubric Weight Sensitivity**  
Our dimension weights (25/20/25/15/15) reflect our pedagogical priorities, but alternative weightings could change relative model rankings. We explore this below.

In [ ]:
# --- Failure Analysis: Where did EduTutor lose? ---
print("=" * 70)
print("\u274c FAILURE ANALYSIS: Scenarios where Base outperformed EduTutor")
print("=" * 70)

# Build comparison pivot
comparison = results_df.pivot_table(
    index=["scenario_id", "profile", "zone", "subject"],
    columns="model",
    values="overall_weighted"
).reset_index()

if "Base Gemma 4" in comparison.columns and "EduTutor" in comparison.columns:
    comparison["delta"] = comparison["EduTutor"] - comparison["Base Gemma 4"]
    failures = comparison[comparison["delta"] < 0].sort_values("delta")
    
    if len(failures) == 0:
        print("\U0001f7e2 EduTutor won or tied on ALL scenarios. No failures to analyze.")
    else:
        print(f"\nEduTutor underperformed on {len(failures)} scenario(s):\n")
        print(f"{'Scenario':<20} {'Profile':<12} {'Zone':<8} {'Base':>6} {'EduT':>6} {'\u0394':>7}")
        print("-" * 65)
        for _, row in failures.iterrows():
            print(f"{row['scenario_id']:<20} {row['profile']:<12} {row['zone']:<8} {row['Base Gemma 4']:>6.2f} {row['EduTutor']:>6.2f} {row['delta']:>+7.2f}")

# --- Sensitivity Analysis: Alternative Weightings ---
print("\n" + "=" * 70)
print("\u2696\ufe0f  SENSITIVITY ANALYSIS: Do results hold under different rubric weights?")
print("=" * 70)

weight_schemes = {
    "Original (25/20/25/15/15)": {"productive_struggle": 0.25, "cognitive_load": 0.20, "emotional_coregulation": 0.25, "profile_adaptation": 0.15, "pedagogical_accuracy": 0.15},
    "Equal weights (20/20/20/20/20)": {"productive_struggle": 0.20, "cognitive_load": 0.20, "emotional_coregulation": 0.20, "profile_adaptation": 0.20, "pedagogical_accuracy": 0.20},
    "Emphasize Emotional (15/15/40/15/15)": {"productive_struggle": 0.15, "cognitive_load": 0.15, "emotional_coregulation": 0.40, "profile_adaptation": 0.15, "pedagogical_accuracy": 0.15},
    "Emphasize Pedagogical (15/15/15/15/40)": {"productive_struggle": 0.15, "cognitive_load": 0.15, "emotional_coregulation": 0.15, "profile_adaptation": 0.15, "pedagogical_accuracy": 0.40},
}

print(f"\n{'Weight Scheme':<40} {'Base':>8} {'EduTutor':>10} {'\u0394':>7} {'Winner':<12}")
print("-" * 80)

sensitivity_results = []
for scheme_name, weights in weight_schemes.items():
    # Recompute weighted scores for all rows
    reweighted = []
    for _, row in results_df.iterrows():
        score = sum(row[dim] * w for dim, w in weights.items())
        reweighted.append({"model": row["model"], "score": score})
    rw_df = pd.DataFrame(reweighted)
    base_mean = rw_df[rw_df["model"] == "Base Gemma 4"]["score"].mean()
    edu_mean = rw_df[rw_df["model"] == "EduTutor"]["score"].mean()
    delta = edu_mean - base_mean
    winner = "\U0001f3c6 EduTutor" if delta > 0 else ("\u2696\ufe0f Tie" if delta == 0 else "Base")
    sensitivity_results.append(delta > 0)
    print(f"{scheme_name:<40} {base_mean:>8.3f} {edu_mean:>10.3f} {delta:>+7.3f} {winner:<12}")

if all(sensitivity_results):
    print("\n\u2705 EduTutor's advantage is robust across all tested weight schemes.")
else:
    print("\n\u26a0\ufe0f  EduTutor's advantage varies under some weight schemes \u2014 see above.")

## Summary

This notebook provides the empirical backbone for our Kaggle submission. Key outputs:

| Output | Purpose |
|---|---|
| `eval_results.csv` | Full per-scenario scoring data |
| `eval_results.png` | Visualization for the video demo |

### Next Step

→ **Notebook 4: `04_agentic_tutor_ui.ipynb`** — Build the interactive demo with Gradio and agentic state tracking.